In [7]:
import os
import pandas as pd
import re
import itertools
import json
import numpy as np

from utilities.loaders import load_results, summarize_results
from argparse import ArgumentParser

In [8]:
# load results.json 
gbt_results = load_results("sample_gbt_results.json")
gbt_results_sum = summarize_results(estimator_name="gbt", results=gbt_results)

In [9]:
gbt_results_sum.shape

(10, 2)

In [10]:
gbt_results_sum

,n_estimators_200|learning_rate_0.01|max_depth_3|verbose_1,n_estimators_400|learning_rate_0.01|max_depth_3|verbose_1
mean_train_acc,0.930686,0.935951
mean_cross_acc,0.924063,0.924971
mean_train_prec,0.823476,0.825523
mean_cross_prec,0.601838,0.566705
mean_train_rec,0.385343,0.451037
mean_cross_rec,0.237054,0.251658
mean_train_f1,0.524952,0.583282
mean_cross_f1,0.302231,0.318331
mean_train_roc_auc,0.688102,0.720246
mean_cross_roc_auc,0.613133,0.619943


In [20]:
lstm_svm_results = load_results("lstm-svm_results.json")
lstm_svm_results_sum = summarize_results("lstm-svm", lstm_svm_results)

In [21]:
lstm_svm_results_sum

,C_1|gamma_0.001|n_a_16|drop_prob_0.05,C_1|gamma_0.001|n_a_16|drop_prob_0.1,C_1|gamma_0.001|n_a_16|drop_prob_0.5,C_1|gamma_0.001|n_a_16|drop_prob_0.75,C_1|gamma_0.001|n_a_32|drop_prob_0.05,C_1|gamma_0.001|n_a_32|drop_prob_0.1,C_1|gamma_0.001|n_a_32|drop_prob_0.5,C_1|gamma_0.001|n_a_32|drop_prob_0.75,C_1|gamma_0.001|n_a_64|drop_prob_0.05,C_1|gamma_0.001|n_a_64|drop_prob_0.1,...,C_1|gamma_0.1|n_a_16|drop_prob_0.1,C_1|gamma_0.1|n_a_16|drop_prob_0.5,C_1|gamma_0.1|n_a_16|drop_prob_0.75,C_1|gamma_0.1|n_a_32|drop_prob_0.05,C_1|gamma_0.1|n_a_32|drop_prob_0.1,C_1|gamma_0.1|n_a_32|drop_prob_0.5,C_1|gamma_0.1|n_a_32|drop_prob_0.75,C_1|gamma_0.1|n_a_64|drop_prob_0.05,C_1|gamma_0.1|n_a_64|drop_prob_0.1,C_1|gamma_0.1|n_a_64|drop_prob_0.5
mean_train_acc,0.498,0.502000,0.498,0.502000,0.498,0.502000,0.500800,0.500800,0.500800,0.500800,...,0.500800,0.500800,0.4992,0.500800,0.500800,0.500800,0.500800,0.4992,0.500800,0.500800
mean_cross_acc,0.498,0.502000,0.498,0.502000,0.498,0.502000,0.500800,0.500800,0.500800,0.500800,...,0.500800,0.500800,0.4992,0.500800,0.500800,0.500800,0.500800,0.4992,0.500800,0.500800
mean_train_prec,0.000,0.502000,0.000,0.502000,0.000,0.502000,0.500800,0.500800,0.500800,0.500800,...,0.500800,0.500800,0.0000,0.500800,0.500800,0.500800,0.500800,0.0000,0.500800,0.500800
mean_cross_prec,0.000,0.502000,0.000,0.502000,0.000,0.502000,0.500800,0.500800,0.500800,0.500800,...,0.500800,0.500800,0.0000,0.500800,0.500800,0.500800,0.500800,0.0000,0.500800,0.500800
mean_train_rec,0.000,1.000000,0.000,1.000000,0.000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.0000,1.000000,1.000000,1.000000,1.000000,0.0000,1.000000,1.000000
mean_cross_rec,0.000,1.000000,0.000,1.000000,0.000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.0000,1.000000,1.000000,1.000000,1.000000,0.0000,1.000000,1.000000
mean_train_f1,0.000,0.668436,0.000,0.668436,0.000,0.668436,0.667366,0.667366,0.667366,0.667366,...,0.667366,0.667366,0.0000,0.667366,0.667366,0.667366,0.667366,0.0000,0.667366,0.667366
mean_cross_f1,0.000,0.668347,0.000,0.668347,0.000,0.668347,0.667196,0.667196,0.667196,0.667196,...,0.667196,0.667196,0.0000,0.667196,0.667196,0.667196,0.667196,0.0000,0.667196,0.667196
mean_train_roc_auc,0.500,0.500000,0.500,0.500000,0.500,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.500000,0.500000,0.5000,0.500000,0.500000,0.500000,0.500000,0.5000,0.500000,0.500000
mean_cross_roc_auc,0.500,0.500000,0.500,0.500000,0.500,0.500000,0.500000,0.500000,0.500000,0.500000,...,0.500000,0.500000,0.5000,0.500000,0.500000,0.500000,0.500000,0.5000,0.500000,0.500000


In [13]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [14]:
svm_hyper_param_config = {'C': 1, 'gamma': 0.001}
lr_hyper_param_config = {'C': 1}
rf_hyper_param_config = {'n_estimators': 200}
gbt_hyper_param_config = {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 3}
svm = SVC(**svm_hyper_param_config, verbose=1)
lr = LogisticRegression(**lr_hyper_param_config, verbose=1)
rf = RandomForestClassifier(**rf_hyper_param_config, verbose=1)
gbt = GradientBoostingClassifier(**gbt_hyper_param_config, verbose=1)